# DQL Live Demons

This notebook is a live teaching walkthrough of Deep Q-Learning (DQL) for limit-order-book execution using `Execution_Methods/rl_lob_execution.py`.

Sign convention used throughout:
- `qty > 0` means **BUY**
- `qty < 0` means **SELL**

Reproducibility and classroom speed:
- We use fixed seeds and small training settings so examples are easier to follow.
- Section 7 uses a compact sensitivity sweep to keep runtime fast


In [37]:
# Setup: imports + project path + helper printing utilities.
from pathlib import Path
import sys
import copy
import random
import numpy as np
import torch
from pprint import pprint

# Properly import packages from different folders/directories.
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "Execution_Methods" / "rl_lob_execution.py").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from Execution_Methods.rl_lob_execution import (
    lob_environment,
    DQN,
    ReplayBuffer,
    get_default_env_params,
    get_default_train_params,
    get_default_eval_params,
    get_epsilon,
    evaluate_policy,
    train_deep_q_execution,
)


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def fmt(x, digits=4):
    if x is None:
        return "None"
    try:
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)


def print_table(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))

    line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    sep = "-+-".join("-" * widths[i] for i in range(len(headers)))
    print(line)
    print(sep)
    for row in rows:
        print(" | ".join(str(cell).ljust(widths[i]) for i, cell in enumerate(row)))


In [38]:
# Baseline configs for all sections (fast classroom defaults).

set_global_seed(42)

env_params = get_default_env_params()
train_params = get_default_train_params()
eval_params = get_default_eval_params()

# Keep deterministic and teaching-friendly.
env_params["seed"] = 42

# Faster defaults for classroom live demos.
train_params["num_episodes"] = 60
train_params["batch_size"] = 64
train_params["buffer_capacity"] = 3000
train_params["target_update_freq"] = 25
train_params["epsilon_decay"] = 180
train_params["parent_qty"] = -10000

eval_params["n_episodes"] = 3

print("Baseline environment params (selected):")
pprint({k: env_params[k] for k in ["reference_price", "tick_size", "n_levels", "seed", "max_steps", "action_fracs"]})
print()
print("Baseline training params (selected):")
pprint({k: train_params[k] for k in ["num_episodes", "batch_size", "buffer_capacity", "gamma", "epsilon_start", "epsilon_end", "epsilon_decay", "target_update_freq", "parent_qty"]})
print()
print("Baseline eval params:")
pprint(eval_params)


Baseline environment params (selected):
{'action_fracs': (0.0,
                  0.1,
                  0.15,
                  0.2,
                  0.25,
                  0.3,
                  0.35,
                  0.4,
                  0.45,
                  0.5,
                  0.55,
                  0.6,
                  0.65,
                  0.7,
                  0.75,
                  0.8,
                  0.85,
                  0.9,
                  0.95,
                  1.0),
 'max_steps': 50,
 'n_levels': 10,
 'reference_price': 28.13,
 'seed': 42,
 'tick_size': 0.01}

Baseline training params (selected):
{'batch_size': 64,
 'buffer_capacity': 3000,
 'epsilon_decay': 180,
 'epsilon_end': 0.05,
 'epsilon_start': 1.0,
 'gamma': 0.99,
 'num_episodes': 60,
 'parent_qty': -10000,
 'target_update_freq': 25}

Baseline eval params:
{'n_episodes': 3}


In [39]:
# Section 1: Problem Framing and Action Space

# Why RL here (in plain language):
# We need to execute a large parent order over time in an uncertain market.
# If we trade too fast, we can move price against ourselves.
# If we trade too slowly, we may run out of time or carry risk.
# DQL learns a policy: "given this market state and remaining inventory, what clip size should I trade now?"

env = lob_environment(**env_params)

print("Execution problem framing:")
print("- Parent order = the full order to complete (example: sell 10,000 shares).")
print("- Child orders = smaller slices sent step by step.")
print("- Goal = finish inventory with low execution cost/slippage.")
print()

print("Action space in this environment:")
print("- The model chooses an action index, not raw shares directly.")
print("- That index maps to a fraction in action_fracs.")
print("- Fraction is applied to REMAINING inventory at each step.")
print()
print("action_fracs:", env.action_fracs)
print("number of discrete actions:", len(env.action_fracs))
print()

remaining_example = 10000
rows = []
for i, frac in enumerate(env.action_fracs):
    abs_clip = int(round(frac * remaining_example))
    sell_signed = -abs_clip
    rows.append([i, f"{frac:.2f}", f"{abs_clip:,}", f"{sell_signed:,}"])

print(f"Example mapping when remaining inventory is {remaining_example:,} shares:")
print_table(rows, headers=["action_idx", "action_frac", "shares this step", "sell signed qty"])
print()
print("What to notice:")
print("- Small fractions = gentler execution (less immediate impact, slower completion).")
print("- Large fractions = aggressive execution (faster completion, potentially higher impact).")
print("- Last step is forced liquidation of all remaining inventory inside env.step().")


Execution problem framing:
- Parent order = the full order to complete (example: sell 10,000 shares).
- Child orders = smaller slices sent step by step.
- Goal = finish inventory with low execution cost/slippage.

Action space in this environment:
- The model chooses an action index, not raw shares directly.
- That index maps to a fraction in action_fracs.
- Fraction is applied to REMAINING inventory at each step.

action_fracs: [0.0, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
number of discrete actions: 20

Example mapping when remaining inventory is 10,000 shares:
action_idx | action_frac | shares this step | sell signed qty
-----------+-------------+------------------+----------------
0          | 0.00        | 0                | 0              
1          | 0.10        | 1,000            | -1,000         
2          | 0.15        | 1,500            | -1,500         
3          | 0.20        | 2,000            | -2,000    

In [40]:
# Section 2: State Design (What the Agent Sees)

# The agent does not see raw Python objects. It sees a numeric state vector.
# This vector contains inventory/time context + top-of-book/depth information.

env = lob_environment(**env_params)
state = env.reset(parent_qty=train_params["parent_qty"])
state = np.asarray(state, dtype=np.float32)

feature_names = [
    "remaining_frac",
    "time_frac",
    "best_bid_rel_ticks",
    "best_ask_rel_ticks",
    "best_bid_size_norm",
    "best_ask_size_norm",
    "bid_price_depth1_rel_ticks",
    "bid_price_depth2_rel_ticks",
    "bid_price_depth3_rel_ticks",
    "bid_size_depth1_log1p",
    "bid_size_depth2_log1p",
    "bid_size_depth3_log1p",
    "ask_price_depth1_rel_ticks",
    "ask_price_depth2_rel_ticks",
    "ask_price_depth3_rel_ticks",
    "ask_size_depth1_log1p",
    "ask_size_depth2_log1p",
    "ask_size_depth3_log1p",
]

print("State vector length:", len(state))
print()
print("Labeled state breakdown:")
for i, (name, val) in enumerate(zip(feature_names, state)):
    print(f"{i:02d}. {name:<30} = {val: .4f}")

print()
print("What to notice:")
print("- remaining_frac and time_frac tell the agent urgency.")
print("- bid/ask relative prices tell where market is vs arrival benchmark.")
print("- size features show liquidity near top of book.")
print("- top-depth features give short horizon structure of the order book.")


State vector length: 18

Labeled state breakdown:
00. remaining_frac                 =  1.0000
01. time_frac                      =  1.0000
02. best_bid_rel_ticks             = -2.0000
03. best_ask_rel_ticks             =  2.0000
04. best_bid_size_norm             =  0.6767
05. best_ask_size_norm             =  0.6986
06. bid_price_depth1_rel_ticks     = -2.0000
07. bid_price_depth2_rel_ticks     = -3.0000
08. bid_price_depth3_rel_ticks     = -4.0000
09. bid_size_depth1_log1p          =  6.7673
10. bid_size_depth2_log1p          =  8.5719
11. bid_size_depth3_log1p          =  9.2090
12. ask_price_depth1_rel_ticks     =  2.0000
13. ask_price_depth2_rel_ticks     =  3.0000
14. ask_price_depth3_rel_ticks     =  4.0000
15. ask_size_depth1_log1p          =  6.9856
16. ask_size_depth2_log1p          =  8.7121
17. ask_size_depth3_log1p          =  9.2108

What to notice:
- remaining_frac and time_frac tell the agent urgency.
- bid/ask relative prices tell where market is vs arrival benchmark.

In [41]:
# Section 3: Reward, Cost, and Transition Mechanics

# This section demonstrates one-step transitions manually.
# In this environment, reward = negative execution cost.
# So higher reward means lower cost (better execution quality).

env = lob_environment(**env_params)
parent_qty = -250
state = env.reset(parent_qty=parent_qty)

print("Episode start:")
print(f"parent_qty={parent_qty}, arrival_price={env.arrival_price:.4f}, max_steps={env.max_steps}")
print()

manual_actions = [1, 6, 12, len(env.action_fracs) - 1]  # from small to aggressive

done = False
for step_i, action_idx in enumerate(manual_actions, start=1):
    if done:
        break
    action_frac = env.action_fracs[action_idx]

    next_state, reward, done, info = env.step(action_idx)

    print(f"Step {step_i}: action_idx={action_idx}, action_frac={action_frac:.2f}")
    print(f"  filled={info['filled']}, remaining={info['remaining']}")
    print(f"  reward={reward:.4f}, step_cost={info['cost']:.4f}, done={done}")
    print()

    state = next_state

print("How to read reward/cost signs:")
print("- reward = -cost by construction.")
print("- positive reward => lower cost (better than benchmark for that step).")
print("- negative reward => higher cost (worse than benchmark for that step).")
print("- if horizon ends with inventory left, env adds terminal mark-to-market penalty.")


Episode start:
parent_qty=-250, arrival_price=28.1300, max_steps=50

Step 1: action_idx=1, action_frac=0.10
  filled=-25, remaining=-225
  reward=-1.0000, step_cost=1.0000, done=False

Step 2: action_idx=6, action_frac=0.35
  filled=-79, remaining=-146
  reward=-3.9500, step_cost=3.9500, done=False

Step 3: action_idx=12, action_frac=0.65
  filled=-95, remaining=-51
  reward=-5.7000, step_cost=5.7000, done=False

Step 4: action_idx=19, action_frac=1.00
  filled=-51, remaining=0
  reward=-3.5700, step_cost=3.5700, done=True

How to read reward/cost signs:
- reward = -cost by construction.
- positive reward => lower cost (better than benchmark for that step).
- negative reward => higher cost (worse than benchmark for that step).
- if horizon ends with inventory left, env adds terminal mark-to-market penalty.


In [42]:
# Section 4: Model and Replay Memory

# DQN approximates Q(s,a): expected future value for each action at state s.
# ReplayBuffer stores past transitions so updates use decorrelated mini-batches.

env = lob_environment(**env_params)
state_dim = len(env.reset(parent_qty=train_params["parent_qty"]))
action_dim = len(env.action_fracs)

policy_net = DQN(state_dim=state_dim, action_dim=action_dim, hidden_dims=train_params["hidden_dims"])
replay_buffer = ReplayBuffer(capacity=32)

print("DQN model:")
print(policy_net)
print()
print(f"state_dim={state_dim}, action_dim={action_dim}")
print()

# Create a few toy transitions from random actions.
state = env.reset(parent_qty=train_params["parent_qty"])
while len(replay_buffer) < 6:
    a = random.randrange(action_dim)
    next_state, reward, done, _ = env.step(a)
    replay_buffer.push(state, a, reward, next_state, done)
    state = env.reset(parent_qty=train_params["parent_qty"]) if done else next_state

batch_size = 4
states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

print("Sampled mini-batch tensor shapes:")
print("states:", tuple(states.shape))
print("actions:", tuple(actions.shape))
print("rewards:", tuple(rewards.shape))
print("next_states:", tuple(next_states.shape))
print("dones:", tuple(dones.shape))

q_out = policy_net(states)
print()
print("DQN output Q-value shape:", tuple(q_out.shape))
print("Interpretation: one row per sampled state, one column per possible action.")


DQN model:
DQN(
  (net): Sequential(
    (0): Linear(in_features=18, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=20, bias=True)
  )
)

state_dim=18, action_dim=20

Sampled mini-batch tensor shapes:
states: (4, 18)
actions: (4,)
rewards: (4,)
next_states: (4, 18)
dones: (4,)

DQN output Q-value shape: (4, 20)
Interpretation: one row per sampled state, one column per possible action.


In [43]:
# Section 5: Training Loop (How Learning Happens)

# Training pipeline in plain language:
# 1) collect transitions from epsilon-greedy interaction
# 2) sample mini-batches from replay buffer
# 3) compute Bellman target and MSE loss
# 4) update policy network
# 5) periodically copy policy -> target network

classroom_env_params = copy.deepcopy(env_params)
classroom_train_params = copy.deepcopy(train_params)
classroom_eval_params = copy.deepcopy(eval_params)

# Keep runtime fast.
classroom_train_params["num_episodes"] = 80
classroom_train_params["epsilon_decay"] = 160
classroom_train_params["target_update_freq"] = 25
classroom_eval_params["n_episodes"] = 3

# Main training call.
dql_result = train_deep_q_execution(
    env_params=classroom_env_params,
    train_params=classroom_train_params,
    eval_params=classroom_eval_params,
    device=None,
    verbose=False,
)

diag = dql_result["training_diagnostics"]
episode_costs = diag.get("episode_costs", [])
loss_history = diag.get("loss_history", [])
final_epsilon = diag.get("final_epsilon")

print("Training completed.")
print("episodes:", len(episode_costs))
print("final_epsilon:", fmt(final_epsilon, 4))
print("sample episode_costs (first 5):", [round(float(x), 4) for x in episode_costs[:5]])
print("sample episode_costs (last 5):", [round(float(x), 4) for x in episode_costs[-5:]])
print("sample losses (first 5):", [round(float(x), 6) for x in loss_history[:5]])
print("sample losses (last 5):", [round(float(x), 6) for x in loss_history[-5:]])

print()
print("What to notice:")
print("- Epsilon shrinks over time: less random exploration, more policy-driven actions.")
print("- Loss tracks Bellman fit quality on sampled transitions.")
print("- Cost trajectory gives practical execution performance signal during learning.")


Training completed.
episodes: 80
final_epsilon: 0.0500
sample episode_costs (first 5): [543.82, 485.52, 426.84, 499.18, 484.4]
sample episode_costs (last 5): [1097.74, 1659.51, 1023.6, 2061.69, 901.05]
sample losses (first 5): [12824.430664, 12804.664062, 15076.275391, 15079.381836, 12584.011719]
sample losses (last 5): [3544.875244, 5132.967285, 2756.331299, 2202.006348, 2389.945312]

What to notice:
- Epsilon shrinks over time: less random exploration, more policy-driven actions.
- Loss tracks Bellman fit quality on sampled transitions.
- Cost trajectory gives practical execution performance signal during learning.


In [44]:
# Section 6: Evaluation and Execution Metrics

# Use training output from Section 5 when available.
# Fallback: do a tiny quick train if this cell is ran directly.
if "dql_result" not in globals() or not isinstance(dql_result, dict):
    quick_train = copy.deepcopy(train_params)
    quick_eval = copy.deepcopy(eval_params)
    quick_train["num_episodes"] = 10
    quick_eval["n_episodes"] = 1
    dql_result = train_deep_q_execution(env_params, quick_train, quick_eval, device=None, verbose=False)

strategy = dql_result.get("strategy")
arrival_vwap = dql_result.get("arrival_vwap")
global_vwap = dql_result.get("global_vwap")
slippage = dql_result.get("slippage")
req_qty = dql_result.get("net_requested_qty")
filled_qty = dql_result.get("net_filled_qty")

print("Strategy:", strategy)
print("arrival_vwap:", fmt(arrival_vwap, 4))
print("global_vwap:", fmt(global_vwap, 4))
print("slippage:", fmt(slippage, 4))
print("net_requested_qty:", req_qty)
print("net_filled_qty:", filled_qty)

if req_qty not in (None, 0) and filled_qty is not None:
    fill_rate = abs(float(filled_qty)) / abs(float(req_qty))
    print("fill_rate:", f"{100.0 * fill_rate:.1f}%")

print()
print("Metric interpretation:")
print("- arrival_vwap: benchmark price near order arrival.")
print("- global_vwap: actual average execution price.")
print("- slippage: gap between actual execution and benchmark.")
print("- net_filled_qty/fill_rate: how completely the policy executes the parent order.")

# Show one episode action schedule for intuition.
eval_diag = dql_result.get("evaluation_diagnostics", {})
episodes = eval_diag.get("episodes", [])
if episodes:
    ep0 = episodes[0]
    print()
    print("Example evaluation episode:")
    print("episode:", ep0.get("episode"))
    print("action_schedule:", [round(float(x), 2) for x in ep0.get("action_schedule", [])])
    print("exec_frac_parent:", [round(float(x), 2) for x in ep0.get("exec_frac_parent", [])])


Strategy: Deep-Q (Single Parent)
arrival_vwap: 28.1300
global_vwap: 28.0265
slippage: 0.1035
net_requested_qty: -10000
net_filled_qty: -10000.0
fill_rate: 100.0%

Metric interpretation:
- arrival_vwap: benchmark price near order arrival.
- global_vwap: actual average execution price.
- slippage: gap between actual execution and benchmark.
- net_filled_qty/fill_rate: how completely the policy executes the parent order.

Example evaluation episode:
episode: 1
action_schedule: [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.0, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4]
exec_frac_parent: [0.1, 0.09, 0.08, 0.07, 0.07, 0.06, 0.05, 0.05, 0.04, 0.04, 0.03, 0.03, 0.03, 0.03, 0.02, 0.02, 0.02, 0.02, 0.01, 0.01, 0.01, 0.0, 0.04, 0.03, 0.02, 0.01, 0.01, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [45]:
# Section 7: Fast Hyperparameter Sensitivity

# We run a tiny directional sweep over:
# - epsilon_decay (exploration schedule)
# - gamma (how far ahead value is considered)
# - action_fracs granularity (coarse vs fine action space)

def run_sensitivity_case(base_env, base_train, base_eval, epsilon_decay, gamma, action_fracs, seed):
    set_global_seed(seed)
    e = copy.deepcopy(base_env)
    t = copy.deepcopy(base_train)
    v = copy.deepcopy(base_eval)

    e["action_fracs"] = tuple(action_fracs)
    t["epsilon_decay"] = int(epsilon_decay)
    t["gamma"] = float(gamma)

    # Keep fast.
    t["num_episodes"] = 8
    t["batch_size"] = min(int(t.get("batch_size", 64)), 32)
    t["buffer_capacity"] = min(int(t.get("buffer_capacity", 3000)), 1200)
    v["n_episodes"] = 2

    out = train_deep_q_execution(e, t, v, device=None, verbose=False)
    ev = out.get("evaluation_diagnostics", {})

    req = out.get("net_requested_qty", t["parent_qty"])
    filled = ev.get("avg_net_filled_qty", out.get("net_filled_qty"))
    fill_rate = None
    if req not in (None, 0) and filled is not None:
        try:
            fill_rate = abs(float(filled)) / abs(float(req))
        except Exception:
            fill_rate = None

    return {
        "epsilon_decay": epsilon_decay,
        "gamma": gamma,
        "granularity": "fine" if len(action_fracs) > 6 else "coarse",
        "slippage": ev.get("avg_slippage", out.get("slippage")),
        "fill_rate": fill_rate,
        "net_filled_qty": filled,
    }

coarse_fracs = (0.0, 0.25, 0.5, 0.75, 1.0)
fine_fracs = tuple(float(x) for x in np.linspace(0.0, 1.0, 11))

cases = []
seed_base = 2026
for gran_name, fracs in [("coarse", coarse_fracs), ("fine", fine_fracs)]:
    for epsilon_decay in [30, 120]:
        for gamma in [0.95, 0.99]:
            cases.append((gran_name, fracs, epsilon_decay, gamma))

sens_rows = []
for i, (gran_name, fracs, eps_d, gam) in enumerate(cases):
    res = run_sensitivity_case(env_params, train_params, eval_params, eps_d, gam, fracs, seed_base + i)
    sens_rows.append(res)

table_rows = []
for r in sens_rows:
    fill_txt = "None" if r["fill_rate"] is None else f"{100.0 * float(r['fill_rate']):.1f}%"
    table_rows.append([
        r["granularity"],
        r["epsilon_decay"],
        f"{float(r['gamma']):.2f}",
        fmt(r["slippage"], 4),
        fill_txt,
        fmt(r["net_filled_qty"], 1),
    ])

print("Fast sensitivity results:")
print_table(table_rows, headers=["granularity", "epsilon_decay", "gamma", "avg_slippage", "fill_rate", "avg_filled_qty"])

# Directional interpretation
print()
print("Directional interpretation (small-run signal, not final production tuning):")

for factor, low, high in [("epsilon_decay", 30, 120), ("gamma", 0.95, 0.99), ("granularity", "coarse", "fine")]:
    low_rows = [r for r in sens_rows if r[factor] == low]
    high_rows = [r for r in sens_rows if r[factor] == high]

    low_slip = np.mean([float(x["slippage"]) for x in low_rows if x["slippage"] is not None])
    high_slip = np.mean([float(x["slippage"]) for x in high_rows if x["slippage"] is not None])

    low_fill_vals = [x["fill_rate"] for x in low_rows if x["fill_rate"] is not None]
    high_fill_vals = [x["fill_rate"] for x in high_rows if x["fill_rate"] is not None]
    low_fill = np.mean(low_fill_vals) if low_fill_vals else np.nan
    high_fill = np.mean(high_fill_vals) if high_fill_vals else np.nan

    verdict = "mixed"
    if high_slip < low_slip and (np.isnan(high_fill) or np.isnan(low_fill) or high_fill >= low_fill):
        verdict = "helped"
    elif high_slip > low_slip and (np.isnan(high_fill) or np.isnan(low_fill) or high_fill <= low_fill):
        verdict = "hurt"

    print(f"- {factor}: high vs low -> slippage {high_slip:.4f} vs {low_slip:.4f}, fill {high_fill:.3f} vs {low_fill:.3f} => {verdict}")

best = sorted(
    sens_rows,
    key=lambda r: (
        float(r["slippage"]) if r["slippage"] is not None else float("inf"),
        -(float(r["fill_rate"]) if r["fill_rate"] is not None else -1.0),
    ),
)[0]

print()
print("Best tiny-run config in this sweep:")
print(best)
print("Use this as a teaching signal only; final tuning should use longer runs and repeated seeds.")


Fast sensitivity results:
granularity | epsilon_decay | gamma | avg_slippage | fill_rate | avg_filled_qty
------------+---------------+-------+--------------+-----------+---------------
coarse      | 30            | 0.95  | 0.0483       | 100.0%    | -10000.0      
coarse      | 30            | 0.99  | 0.0386       | 100.0%    | -10000.0      
coarse      | 120           | 0.95  | 0.2676       | 100.0%    | -10000.0      
coarse      | 120           | 0.99  | 0.0399       | 100.0%    | -10000.0      
fine        | 30            | 0.95  | 0.0399       | 100.0%    | -10000.0      
fine        | 30            | 0.99  | 0.0870       | 100.0%    | -10000.0      
fine        | 120           | 0.95  | 0.0428       | 100.0%    | -10000.0      
fine        | 120           | 0.99  | 0.2578       | 100.0%    | -10000.0      

Directional interpretation (small-run signal, not final production tuning):
- epsilon_decay: high vs low -> slippage 0.1520 vs 0.0534, fill 1.000 vs 1.000 => hurt
- gamma: h